# WarpTracer benchmarks

Select **Runtime → Change runtime type → GPU**, then run setup, benchmark, and results.
Choose one preset: `navigation`, `batches`, `fusion`, or `quick`.
Presets set car counts, workloads, durations and output paths; no other settings need editing.
Navigation compares the LiDAR baseline with disparity control. GPU presets stop immediately if CUDA is unavailable.
The `quick` preset also works on CPU. Output streams into the cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch",
                    "performance-prototype", "https://github.com/yuywe/WarpTracer.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "fetch", "origin", "performance-prototype:refs/remotes/origin/performance-prototype"], check=True)
if subprocess.run(["git", "show-ref", "--verify", "--quiet", "refs/heads/performance-prototype"]).returncode:
    subprocess.run(["git", "branch", "--no-track", "performance-prototype", "origin/performance-prototype"], check=True)
subprocess.run(["git", "switch", "performance-prototype"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "performance-prototype"], check=True)
os.environ["UV_PYTHON"] = sys.executable
subprocess.run(["uv", "sync", "--locked"], check=True)

def run_stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
        except KeyboardInterrupt:
            import signal
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
            raise


In [ ]:
preset = "navigation"  # or "batches", "fusion", "quick"
run_stream(["uv", "run", "warptracer", "benchmark", preset])


In [ ]:
import json
report = json.loads(Path(f"outputs/{preset}.json").read_text())
print(report["device_name"], report["physics_engine"])
for result in report["results"]:
    s = result["summary"]
    print("cars:", result["environments"], result["case"], result["integrator"], result["backend"],
          round(s["median_aggregate_environment_transitions_per_second"]), "aggregate transitions/s;",
          round(s["median_batch_transitions_per_second"]), "batch transitions/s;",
          "variable timing" if s["timing_variable"] else "max/min <= 1.2")
print("Full report:", Path(f"outputs/{preset}.json").resolve())
